# 5. Measure BEFORE vs AFTER

**Goal:** learn the lab's most important habit.

> Never call something an optimization without a measured baseline.

The loop is always:

```
B0 baseline -> implement candidate -> B1 -> same protocol -> delta -> verdict
```

## Setup

In [1]:
from pathlib import Path
import sys

REPO = Path.cwd()
while not (REPO / "packages" / "deepseek_flash_v4_mini").exists():
    if REPO == REPO.parent:
        raise RuntimeError("Run this notebook from inside deepseek_flash_lab")
    REPO = REPO.parent

sys.path.insert(0, str(REPO / "packages"))
sys.path.insert(0, str(REPO))
import torch

torch.manual_seed(0)
print("repo root:", REPO.name)
print("torch:", torch.__version__)

repo root: deepseek_flash_lab
torch: 2.13.0+cu130


## 1. Load a stored experiment

The RoPE experiment compares two mathematically equivalent formulations.

In [2]:
import json

baseline_dir = REPO / "results" / "v4" / "rope_formulation" / "A0_complex"
candidate_dir = REPO / "results" / "v4" / "rope_formulation" / "A1_rotate_half"

b = json.loads((baseline_dir / "metrics.json").read_text())
c = json.loads((candidate_dir / "metrics.json").read_text())
print("baseline keys:", sorted(b))

baseline keys: ['active_parameters', 'forward_ms', 'p50_latency_ms', 'p95_latency_ms', 'parameters', 'peak_cpu_memory_mb', 'peak_gpu_memory_mb', 'tokens_per_second', 'trainable_parameters']


## 2. Build the delta table

`bench.delta` knows which direction is good for each metric. Latency: lower is
better. Throughput: higher is better.

In [3]:
from bench.delta import compare_metrics, format_table, verdict

rows = compare_metrics(b, c)
print(format_table(rows, candidate_label="A1"))
print()
print(verdict(rows))

| Metric                       |     Baseline |           A1 |   Absolute Δ |  Improve % | Better? |
| ---------------------------- | ------------: | ------------: | ------------: | ----------: | ------- |
| active_parameters            |            0 |            0 |           +0 |          — |       — |
| forward_ms                   |     0.009653 |      0.06742 |     +0.05776 |   -598.41% |      NO |
| p50_latency_ms               |     0.009367 |      0.06149 |     +0.05212 |   -556.41% |      NO |
| p95_latency_ms               |      0.01021 |       0.0882 |     +0.07799 |   -763.91% |      NO |
| parameters                   |            0 |            0 |           +0 |          — |       — |
| peak_cpu_memory_mb           |          618 |        618.3 |       +0.258 |     -0.04% |      NO |
| peak_gpu_memory_mb           |            0 |            0 |           +0 |          — |    SAME |
| tokens_per_second            |    2.652e+07 |    3.797e+06 |   -2.272e+07 |    -85.68

**Notice:** `Improve %` is negative for latency because A1 is slower.
The verdict is `REGRESSION`. That is a successful measurement, not a failure.

## 3. Reproduce a timing in memory

We can time the two functions directly without writing any files.
`time_ms` runs warmup iterations first, then measures.

In [4]:
from bench.latency import time_ms
from deepseek_flash_v4_mini.model.rope import apply_rope, rope_freqs, rope_reference_rotate_half

torch.manual_seed(42)
q = torch.randn(4, 4, 64, 16)
freqs = rope_freqs(16, 64, q.device)

a0 = time_ms(lambda: apply_rope(q, freqs=freqs), warmup=5, iters=20)
a1 = time_ms(
    lambda: rope_reference_rotate_half(q, freqs.real, freqs.imag),
    warmup=5, iters=20,
)
print("A0 complex     mean_ms:", round(a0["mean_ms"], 5))
print("A1 rotate-half mean_ms:", round(a1["mean_ms"], 5))
print("A1 is faster?", a1["mean_ms"] < a0["mean_ms"])

A0 complex     mean_ms: 0.00958
A1 rotate-half mean_ms: 0.0609
A1 is faster? False


**Important:** timings vary between machines and runs. That is exactly why the
protocol requires the same hardware, batch, sequence, seed, and warmup. A
single run is not a scientific claim.

## 4. Numerical correctness

A candidate can be slower and still be useful if it is more accurate, or
faster but wrong. Always check both speed and correctness.

In [5]:
y0 = apply_rope(q, freqs=freqs)
y1 = rope_reference_rotate_half(q, freqs.real, freqs.imag)
print("max abs error :", (y0 - y1).abs().max().item())
print("cosine sim    :", torch.nn.functional.cosine_similarity(
    y0.flatten(), y1.flatten(), dim=0
).item())

max abs error : 0.0
cosine sim    : 1.0


## Recap

- Record a baseline before changing code.
- Compare with the **same** protocol.
- Report the delta honestly, including regressions.
- Check numerics as well as speed.
- Save the result bundle and update the history CSV.

**Try it:** run `python scripts/run_rope_delta_demo.py` from the repo root and
open `results/v4/rope_formulation/optimization_report.md`.